In [0]:
from pyspark.sql.functions import split, trim, col, expr

# Filter out null values from main order tables
sdf_filtered = spark.read.table('kfcus_digital_prod.mparticle_events.order_details_rs').filter("order_products IS NOT NULL")

# Prepare ordered items as a list of unique string and stripped of spaces
sdf_prepared = sdf_filtered.withColumn('order_product_list', expr("array_distinct(transform(split(order_products, ','), x -> trim(x)))"))
sdf_prepared = sdf_prepared.withColumn('order_product_list', col('order_product_list').cast('array<string>'))



In [0]:
# Data cleaning of items we don't want to recommend
sauce_list = [
    "KFC Sauce",
    "Honey Sauce",
    "Classic Ranch",
    "Hot Sauce",
    "Honey BBQ",
    "Honey Mustard",
    "Buffalo Ranch",
    "Comeback Sauce",
    "Ketchup",
    "Sticky Chicky Sweet 'n Sour Sauce",
    "Cheese Sauce",
    "Ranch",
    "BBQ",
    "Honey",
    "jalapeno pesto",
    "thai sweet 'n spicy",
    "creole honey mustard",
    "Sticky Chicky Sweet 'n Sour Sauce"
]

utensils = [
    "1 Paper Plate",
    "Paper Plate",
    "Fork",
    "Spoon",
    "Knife",
    "Straw"
]

miscellaneous = ["Bag Fee", "$1 KFC Foundation Donation", "$1 Add Hope Donation", "undefined"]

free_items = ['FREE Beverage Bucket - 12 pc. Chicken Meal',
 '12 pc. Meal (Variety) with 6 FREE Cookies',
 'FREE Beverage Bucket - 12pc Meal (Variety)',
 '16 pc. Meal (Variety) with 6 FREE Cookies',
 'FREE Beverage Bucket - 16 pc. Chicken Meal',
 'FREE Beverage Bucket - 16 pc Meal (Variety)',
 '12 pc. Meal (Drum & Thigh) with 6 FREE Cookies',
 '12pc Meal (Variety) with FREE Beverage Bucket',
 'FREE Beverage Bucket - 12pc Meal (Drum & Thigh)',
 '12 pc. Tenders Meal with 6 FREE Cookies',
 '16pc Meal (Variety) with FREE Beverage Bucket',
 '8 pc. Chicken Meal + FREE 12 pc. Nuggets',
 'FREE Beverage Bucket - 12 pc. Tenders Meal',
 '16 pc. Tenders Meal with 6 FREE Cookies',
 'FREE Beverage Bucket - 12pc Tenders Meal',
 'FREE Beverage Bucket - 16pc Tenders Meal',
 '12pc Meal (Drum & Thigh) with FREE Beverage Bucket',
 'FREE Beverage Bucket - 16 pc. Tenders Meal',
 '8 pc. Tenders Meal + FREE 12 pc. Nuggets',
 '16pc Tenders Meal with FREE Beverage Bucket',
 '12pc Tenders Meal with FREE Beverage Bucket',
 'FREE Beverage Bucket - 20pc Tenders Meal',
 '20pc Tenders Meal with FREE Beverage Bucket',
 '12pc Meal (Drums & Thighs) with FREE Beverage Bucket']

# Remove items from our specified list
items_to_remove = sauce_list + utensils + miscellaneous + free_items
sdf_cleaned = sdf_prepared.withColumn(
    'order_product_list', 
    expr("filter(order_product_list, x -> NOT array_contains(array({}), x))".format(', '.join(["'{}'".format(item.replace("'", "''")) for item in items_to_remove])))
)

# Drop the empty strings.
sdf_cleaned = sdf_cleaned.withColumn('order_product_list', expr("filter(order_product_list, x -> x != '')"))

sdf_cleaned = sdf_cleaned.filter(expr("size(order_product_list) > 0"))

In [0]:
# UDF to map product names to slugs and drop items not in the mapping
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

product_slug_sdf = spark.read.table('kfcus_digital_prod.product_data.product_slug_mapping')

# Create a dictionary for product name to slug mapping
product_slug_dict = {row['Menu Item Name']: row['Slug'] for row in product_slug_sdf.collect()}

def map_to_slug(product_list):
    return [product_slug_dict[product] for product in product_list if product in product_slug_dict]

map_to_slug_udf = udf(map_to_slug, ArrayType(StringType()))

# Apply the UDF to convert product names to slugs
sdf_cleaned_mapped = sdf_cleaned.withColumn('order_product_list', map_to_slug_udf(col('order_product_list')))

# Drop any row with an empty order_product_list
sdf_cleaned_mapped = sdf_cleaned_mapped.filter(expr("size(order_product_list) > 0"))

# sdf_cleaned_mapped.display()
# sdf_cleaned_mapped.count()

In [0]:
# Save our table as a delta table for 01_market_basket_analysis and 02_collaborative filter
catalog = 'kfcus_digital_model'
schema = 'recommender_test'
sdf_cleaned_mapped.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{catalog}.{schema}.cleaned_mapped_dataset')